# Defter 7 — Hakem eksiklerinin kapatılması
**Embedding tree growth theory into AI** · Prof. Dr. İlker Ercanlı

Makale v2'nin bağımsız denetiminde kalan eksikler:
* **G1** XGBoost ve doğrusal karma modelin iklim senaryolarına tepkisi (sinir ağlarıyla aynı 50.000 halka).
* **G2** Ampirik karşılaştırma: halkalardan doğrudan hesaplanan alan düzeyi iklim duyarlılığı ile model tepkisinin yönü.
* **G3** Tohum değişkenliğini içeren güven aralıkları; eşdeğerlik sınırı ±0,01 / 0,02 / 0,03.
* **G4** Ayarlanmış XGBoost (rastgele arama + 6 milyon eğitim halkası).
* **G5** m'nin sağlamlığı: farklı başlangıç değerleri ve serbest ağdan üretilmiş sentetik veri.
* **G6** En kurak alanlar: Kuzey Amerika içi ve dışı.

Tablolar `ANALIZ_SON/G1…G6_*.csv`. **Çalışma zamanı:** T4 GPU, "Yüksek RAM" açık. Süre ≈ 2,5–3,5 saat.
Kesilirse aynı defteri yeniden "Tümünü çalıştır": bitmiş işler atlanır.

In [ ]:
# --- Kurulum ---
import os, json, time, math, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
try:
    from google.colab import drive
    drive.mount('/content/drive')
    ANA = '/content/drive/MyDrive/ITRDB_AI'
except ImportError:
    ANA = os.environ.get('ITRDB_AI', '.')
SONUC = os.path.join(ANA, 'SONUC_SON'); os.makedirs(SONUC, exist_ok=True)
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Cihaz:', DEV, torch.cuda.get_device_name(0) if DEV == 'cuda' else '(GPU yok: Çalışma zamanı → T4 GPU seçin)')

AYAR = dict(batch=32768, max_epoch=25, sabir=3, lr=2e-3, lr_gizil=1e-2, J=8, gizli=128, emb=8,
            takson_dusur=0.10, su_onsel=0.5, tau_onsel=0.5, sv_onsel=0.5, desc_gurultu=0.2, wd=1e-4,
            rastgele_etki=False,   # False: popülasyon modeli (ana analiz); True: ağaç/alan rastgele etkili duyarlılık analizi
            r0_bas_cm=2.0, sifir_mm=0.005, hizli_deneme=False)

In [ ]:
# --- Veri ---
d = np.load(os.path.join(ANA, 'TENSOR', 'veri_v2.npz'))
soz = json.load(open(os.path.join(ANA, 'TENSOR', 'veri_v2_sozluk.json')))
YIL0 = soz['yil0']; S = len(soz['site_code']); NG = len(soz['genus']); NS = len(soz['species'])
NSER = len(d['series_site']); NTREE = int(d['series_tree'].max()) + 1
t = lambda x, dt=None: torch.as_tensor(np.asarray(x), dtype=dt).to(DEV)

# Halkalar: 1901 halkaları çıkar (önceki yılın iklimi yok)
k = d['ring_year'] > YIL0
R_ser = t(d['ring_series'][k], torch.long); R_yil = t(d['ring_year'][k], torch.long)
R_w = t(d['ring_rw'][k], torch.float32); R_rp = t(d['ring_rprev'][k], torch.float32) / 10   # cm
ser_site = t(d['series_site'], torch.long); ser_tree = t(d['series_tree'], torch.long)
ser_zaman = t(d['series_zaman'], torch.bool)
site_genus = t(d['site_genus'], torch.long); site_species = t(d['site_species'], torch.long)
NR = len(R_w)

# Aylık iklim özellikleri [alan, ay(124*12), kanal]: T, log(1+P), M, VPD, gE
clim = d['clim']; soil = d['soil']; gE = d['gE']
NY = clim.shape[1]
ozel = np.stack([clim[..., 0], np.log1p(np.maximum(clim[..., 1], 0)), soil, clim[..., 3],
                 np.broadcast_to(gE[:, None, :], soil.shape)], -1).reshape(S, NY * 12, 5).astype(np.float32)
OZ = t(ozel); del ozel, clim
oz_ort = OZ.reshape(-1, 5).mean(0); oz_ss = OZ.reshape(-1, 5).std(0)
desc = d['desc']; DESC = t((desc - desc.mean(0)) / desc.std(0), torch.float32)
# Yetişme ortamı: 11 sürekli değişken (standartlaştırılmış) + biyom, ekolojik bölge, Köppen sınıfları (son indeks = bilinmeyen)
env = d['env']; ENV = t((env - env.mean(0)) / env.std(0), torch.float32)
site_biyom = t(d['site_biyom'], torch.long); site_eko = t(d['site_eko'], torch.long); site_kg = t(d['site_kg'], torch.long)
NB, NE, NK = soz['n_biyom'], soz['n_eko'], soz['n_kg']
KAT_GOR = {}   # bölünmeye göre eğitimde görülen sınıflar (görülmeyenler 'bilinmeyen' gömmesine gider)
ORTAM_BOYUT = ENV.shape[1] + 3 * 4

class OrtamGirdi(nn.Module):
    """Alan başına yetişme ortamı vektörü: sürekli değişkenler + biyom/ekolojik bölge/Köppen gömmeleri (her biri 4 boyut)"""
    def __init__(self):
        super().__init__()
        self.b = nn.Embedding(NB + 1, 4); self.e = nn.Embedding(NE + 1, 4); self.k = nn.Embedding(NK + 1, 4)
    def forward(self, G):
        b, e, k = G['b'], G['e'], G['k']
        if KAT_GOR:
            b = torch.where(KAT_GOR['b'][b], b, torch.full_like(b, NB)); e = torch.where(KAT_GOR['e'][e], e, torch.full_like(e, NE))
            k = torch.where(KAT_GOR['k'][k], k, torch.full_like(k, NK))
        if self.training:      # %10 rastgele maskeleme: 'bilinmeyen' gömmesi de öğrenilsin
            m = torch.rand(len(b), device=DEV) < 0.10
            b = torch.where(m, torch.full_like(b, NB), b); e = torch.where(m, torch.full_like(e, NE), e); k = torch.where(m, torch.full_like(k, NK), k)
        x = G['env'] + (AYAR['desc_gurultu'] * torch.randn_like(G['env']) if self.training else 0)
        return torch.cat([x, self.b(b), self.e(e), self.k(k)], 1)
print(f'{NR:,} halka, {NSER:,} seri, {NTREE:,} ağaç, {S} alan — veri {"GPU" if DEV=="cuda" else "CPU"} belleğinde')

# Yerel uyum için alan nem normali: 1901–1960 aylık toprak nemi ortalaması (zamansal testte de yalnız eğitim dönemi)
MBAR = OZ[:, :60 * 12, 2].mean(1); MREF = MBAR.mean()
def alan_girdi(s):
    """Alan düzeyi girdiler (SHAP ve ihlal analizinde gruplar halinde değiştirilebilsin diye sözlük)"""
    return dict(g=site_genus[s], sp=site_species[s], desc=DESC[s], env=ENV[s], b=site_biyom[s], e=site_eko[s], k=site_kg[s], mbar=MBAR[s])

def girdiler(idx):
    """Halka düzeyi girdiler: 24 aylık iklim penceresi X, birikmiş yarıçap rp, seri ve alan girdileri"""
    X, s = pencere(idx); G = alan_girdi(s); G.update(X=X, rp=R_rp[idx], ser=R_ser[idx]); return G

def pencere(idx):
    """Halka idx için 24 aylık pencere (önceki yıl Ocak – bu yıl Aralık): [B, 24, 5]"""
    s = ser_site[R_ser[idx]]
    bas = (R_yil[idx] - 1 - YIL0) * 12
    ay = bas[:, None] + torch.arange(24, device=DEV)[None, :]
    return OZ[s[:, None], ay], s

In [ ]:
# --- Test bölünmeleri ---
site_bolge = np.array(soz['site_bolge']); site_kat = d['site_blok_kat']; site_takson = d['site_takson_test']
site_ic = d['site_ic_dogrulama']

def bolunme(ad):
    """Dönüş: (eğitim, doğrulama, test) halka indeksleri ve test alanlarında takson gizlensin mi"""
    site_r = ser_site[R_ser].cpu().numpy(); yil = R_yil.cpu().numpy(); zuy = ser_zaman[R_ser].cpu().numpy()
    gizle = False
    if ad == 'zaman':
        test = zuy & (yil > 1960); havuz = yil <= 1960
    elif ad.startswith('bolge:'):
        tb = np.isin(site_r, np.where(site_bolge == ad.split(':')[1])[0]); test = tb; havuz = ~tb
    elif ad.startswith('blok:'):
        tb = np.isin(site_r, np.where(site_kat == int(ad.split(':')[1]))[0]); test = tb; havuz = ~tb
    elif ad == 'takson':
        tb = np.isin(site_r, np.where(site_takson)[0]); test = tb; havuz = ~tb; gizle = True
    elif ad.startswith('iklim:'):
        # İklim dışa-genelleme: en sıcak (1901–1960 ort. sıcaklık) ya da en kurak (P/PET) %10 alan yalnız testte
        tur = ad.split(':')[1]
        if tur == 'kurak2':          # %10–20 kurak bant test; eğitim yalnız en kurak %20 dışı alanlarla
            x = -desc[:, 5]; q8, q9 = np.quantile(x, [0.8, 0.9])
            test = np.isin(site_r, np.where((x >= q8) & (x < q9))[0]); havuz = np.isin(site_r, np.where(x < q8)[0])
        else:
            x = {'sicak': desc[:, 2], 'kurak': -desc[:, 5], 'nemli': desc[:, 5]}[tur]
            tb = np.isin(site_r, np.where(x >= np.quantile(x, 0.9))[0]); test = tb; havuz = ~tb
    else:
        raise ValueError(ad)
    ic = site_ic[site_r]
    egit = np.where(havuz & ~ic)[0]; dog = np.where(havuz & ic)[0]; test = np.where(test)[0]
    return t(egit, torch.long), t(dog, torch.long), t(test, torch.long), gizle

In [ ]:
# --- Modeller ---
class AlanAgi(nn.Module):
    """z_s -> parametreler. Cins/tür gömmeleri + standartlaştırılmış alan tanımlayıcıları."""
    def __init__(self, n_cikis, emb, gizli, sutun=None, ortam=False):
        super().__init__()
        self.ortam = OrtamGirdi() if ortam else None
        self.sutun = list(range(DESC.shape[1])) if sutun is None else sutun   # saf teori: yalnız [enlem, yükselti]
        self.ge = nn.Embedding(NG + 1, emb); self.se = nn.Embedding(NS + 1, emb)   # son indeks = "bilinmeyen takson"
        self.mlp = nn.Sequential(nn.Linear(2 * emb + len(self.sutun) + (ORTAM_BOYUT if ortam else 0), gizli), nn.GELU(),
                                 nn.Linear(gizli, gizli), nn.GELU(), nn.Linear(gizli, n_cikis))
    def forward(self, G, gizle_maske=None):
        g, sp = G['g'], G['sp']
        if gizle_maske is not None:
            g = torch.where(gizle_maske, torch.full_like(g, NG), g); sp = torch.where(gizle_maske, torch.full_like(sp, NS), sp)
        d = G['desc'][:, self.sutun]
        d = d + (AYAR['desc_gurultu'] * torch.randn_like(d) if self.training else 0)
        parcalar = [self.ge(g), self.se(sp), d] + ([self.ortam(G)] if self.ortam is not None else [])
        return self.mlp(torch.cat(parcalar, 1))

def monoton(x, a, c, s):
    """Denklem (4): negatif olmayan lojistik karışımı, 0–1 arası, azalmayan. x:[B,24], a,c,s:[B,J]"""
    w = F.softplus(a) + 1e-4
    z = torch.sigmoid((x[..., None] - c[:, None, :]) / s[:, None, :])
    return (z * w[:, None, :]).sum(-1) / w.sum(-1, keepdim=True)

def alan_gor_bul(seri_gor):
    m = torch.zeros(S, dtype=torch.bool, device=DEV); m[ser_site[seri_gor]] = True
    return m

class Gizil(nn.Module):
    """Seri başına gizil başlangıç yarıçapı r0 = softplus(ρ) ve ağaç etkisi u; hiyerarşik önsel cezası"""
    def __init__(self, r0_bas):
        super().__init__()
        rho0 = math.log(math.expm1(r0_bas))
        self.rho = nn.Parameter(torch.full((NSER,), rho0, device=DEV))
        self.mu_g = nn.Parameter(torch.full((NG + 1,), rho0, device=DEV))
        self.u = nn.Parameter(torch.zeros(NTREE, device=DEV))          # ağaç etkisi
        self.v = nn.Parameter(torch.zeros(S, device=DEV))              # alan etkisi (toprak, sıklık vb. kaydedilmemiş farklar)
        # Önsel ölçekleri sabit (zayıf bilgili) hiperparametreler: MAP ile öğrenilen varyans bileşenleri sıfıra çöker.
        # Yeni ağaç/seri için öngörüde kullanılan yayılım ise eğitilmiş ağaçların GÖRGÜL dağılımından alınır.
        self.su_onsel, self.tau_onsel = AYAR['su_onsel'], AYAR['tau_onsel']
        self.register_buffer('su_gorgul', torch.tensor(AYAR['su_onsel'], device=DEV))
        self.register_buffer('tau_gorgul', torch.tensor(AYAR['tau_onsel'], device=DEV))
        self.sv_onsel = AYAR['sv_onsel']
        self.register_buffer('sv_gorgul', torch.tensor(AYAR['sv_onsel'], device=DEV))
        self.register_buffer('ort_u', torch.tensor(0., device=DEV)); self.register_buffer('ort_v', torch.tensor(0., device=DEV))
        self.register_buffer('ort_d', torch.tensor(0., device=DEV))
    def su(self): return self.su_gorgul
    def tau(self): return self.tau_gorgul
    def gorgul_guncelle(self, seri_gor, agac_gor):
        with torch.no_grad():
            g = site_genus[ser_site]
            # Görgül dağılım: ortalama da alınır (ağın sabit terimi ile rastgele etkilerin ortalaması birbirine kayabilir)
            u, d, v = self.u[agac_gor], (self.rho - self.mu_g[g])[seri_gor], self.v[alan_gor_bul(seri_gor)]
            self.su_gorgul.copy_(u.std().clamp_min(0.02)); self.ort_u.copy_(u.mean())
            self.tau_gorgul.copy_(d.std().clamp_min(0.02)); self.ort_d.copy_(d.mean())
            self.sv_gorgul.copy_(v.std().clamp_min(0.02)); self.ort_v.copy_(v.mean())
    def ceza(self, seri_gor, agac_gor):
        """Hiyerarşik önsel; YALNIZ eğitimde verisi olan seri ve ağaçlar üzerinden"""
        g = site_genus[ser_site]
        d = (self.rho - self.mu_g[g])[seri_gor]; u = self.u[agac_gor]; v = self.v[alan_gor_bul(seri_gor)]
        return ((d ** 2).sum() / (2 * self.tau_onsel ** 2) + (u ** 2).sum() / (2 * self.su_onsel ** 2)
                + (v ** 2).sum() / (2 * self.sv_onsel ** 2))
    def gizil_deger(self, ser, ornek=None):
        """Halka başına ρ ve u. ornek=(seri_gor, agac_gor, gizle): görülmemişler için önselden rastgele çekim"""
        rho, u, v = self.rho[ser], self.u[ser_tree[ser]], self.v[ser_site[ser]]
        if ornek is not None:
            seri_gor, agac_gor, gizle, mu_glob = ornek
            g = site_genus[ser_site[ser]]
            mu = torch.where(gizle, mu_glob, self.mu_g[g]) if gizle is not None else self.mu_g[g]
            rho = torch.where(seri_gor[ser], rho, mu + self.ort_d + self.tau() * torch.randn_like(rho))
            u = torch.where(agac_gor[ser_tree[ser]], u, self.ort_u + self.su() * torch.randn_like(u))
            v = torch.where(alan_gor_bul(seri_gor)[ser_site[ser]], v, self.ort_v + self.sv_gorgul * torch.randn_like(v))
        return rho, u + v

class TeoriModeli(nn.Module):
    def __init__(self, J=8, emb=8, gizli=128, ablasyon=None, r0_bas=2.0, m_bas=2/3, vpd=False, m_sabit=None, saf=False, ortam=False, uyum=False, genis=False, tur=False):
        super().__init__()
        self.J, self.abl = J, ablasyon
        self.uyum = uyum
        self.mgen = 0.3 if genis else 0.1   # nem eşiği merkezlerinin kayma aralığı
        self.m_logit = math.log(m_bas / (1 - m_bas)); self.vpd = vpd; self.m_sabit = m_sabit
        n = 3 + 6 * J + 24 + 1 + 1          # +1: başlangıç yarıçapı r0 (alan ve takson düzeyinde, ağdan)
        if ablasyon == 'abl_min': n += 3
        if vpd: n += 3 * J                 # VPD yanıtı (üçüncü sınırlayıcı faktör)
        if uyum: n += 1                    # yerel uyum katsayısı λ
        self.Dbaz = torch.linspace(0.2, 3.0, J, device=DEV)
        self.ag = AlanAgi(n, emb, gizli, sutun=[] if tur else ([0, 1] if saf else None), ortam=ortam)   # saf: iklim normalleri parametre ağına girmez
        if ablasyon == 'abl_mono':      # monoton yanıt yerine kısıtsız küçük ağ (girdi: x ve alan parametreleri)
            self.serbT = nn.Sequential(nn.Linear(1 + 3 * J, 32), nn.GELU(), nn.Linear(32, 1))
            self.serbM = nn.Sequential(nn.Linear(1 + 3 * J, 32), nn.GELU(), nn.Linear(32, 1))
        self.gz = Gizil(r0_bas)
        self.Tbaz = torch.linspace(-5, 25, J, device=DEV); self.Mbaz = torch.linspace(0.02, 0.6, J, device=DEV)

    def parametreler(self, G, gizle=None):
        if torch.is_tensor(G): G = alan_girdi(G)
        o = self.ag(G, gizle); J = self.J
        p = {}
        p['eta'] = torch.exp(o[:, 0] + math.log(0.7))
        p['Bmax'] = torch.exp(o[:, 1] + math.log(1e4))                         # cm² (başlangıç: r ≈ 56 cm)
        p['m'] = torch.sigmoid(o[:, 2] + self.m_logit)                            # başlangıç varsayılan 2/3
        if self.m_sabit is not None: p['m'] = torch.full_like(p['m'], self.m_sabit)   # profil olabilirlik: m sabit
        if getattr(self, 'm_cins', None) is not None:   # sentetik veri üretimi: m cinse göre dışarıdan; η aynı ortalama büyümeyi korur
            m_yeni = self.m_cins[G['g']]; p['eta'] = p['eta'] * 300.0 ** (p['m'] - m_yeni); p['m'] = m_yeni
        i = 3
        p['aT'], p['cT'], p['sT'] = o[:, i:i+J], self.Tbaz + 5 * torch.tanh(o[:, i+J:i+2*J]), 0.5 + 2 * F.softplus(o[:, i+2*J:i+3*J]); i += 3 * J
        p['aM'], p['cM'], p['sM'] = o[:, i:i+J], self.Mbaz + self.mgen * torch.tanh(o[:, i+J:i+2*J]), 0.005 + 0.03 * F.softplus(o[:, i+2*J:i+3*J]); i += 3 * J
        p['omega'] = torch.softmax(o[:, i:i+24], 1); i += 24
        if self.vpd:
            p['aD'], p['cD'], p['sD'] = o[:, i:i+J], self.Dbaz + 0.3 * torch.tanh(o[:, i+J:i+2*J]), 0.05 + 0.3 * F.softplus(o[:, i+2*J:i+3*J]); i += 3 * J
        p['log_sigma'] = o[:, i] - 0.3; i += 1
        p['r0'] = F.softplus(o[:, i] + math.log(math.expm1(AYAR['r0_bas_cm']))); i += 1   # cm
        if self.abl == 'abl_min': p['karisim'] = torch.softmax(o[:, i:i+3], 1)
        if self.uyum: p['lam'] = torch.sigmoid(o[:, -1])          # 0 = mutlak tepki, 1 = tamamen yerel uyumlu
        return p

    def forward(self, idx, gizle=None, ornek=None, G=None):
        G = girdiler(idx) if G is None else G
        X = G['X']; p = self.parametreler(G, gizle); G0 = G
        ser = G['ser']
        rho_i, u_i = self.gz.gizil_deger(ser, ornek)
        if not AYAR['rastgele_etki']: u_i = torch.zeros_like(u_i)
        r0 = 0.05 if self.abl == 'abl_r0' else (F.softplus(rho_i) if AYAR['rastgele_etki'] else p['r0'])
        r = r0 + G['rp']; B = math.pi * r ** 2                                   # cm²
        # Denklem (2): büyüme potansiyeli
        if self.abl == 'abl_G':
            G = p['eta'] * 30.0
        else:
            x = (B / p['Bmax']).clamp_min(1e-8) ** (1 - p['m'])
            G = p['eta'] * B.clamp_min(1e-6) ** p['m'] * F.softplus(20 * (1 - x)) / 20
        # Denklem (3)–(4): sınırlayıcı faktör
        T, M, gEk = X[..., 0], X[..., 2], X[..., 4]
        if self.uyum: M = M - p['lam'][:, None] * (G0['mbar'][:, None] - MREF)   # eşik alanın nem normaline göre kayar
        if self.abl == 'abl_mono':
            pT = torch.cat([p['aT'], p['cT'], p['sT']], 1); pM = torch.cat([p['aM'], p['cM'], p['sM']], 1)
            gT = torch.sigmoid(self.serbT(torch.cat([T[..., None] / 10, pT[:, None, :].expand(-1, 24, -1)], -1))[..., 0])
            gM = torch.sigmoid(self.serbM(torch.cat([M[..., None] * 5, pM[:, None, :].expand(-1, 24, -1)], -1))[..., 0])
        else:
            gT = monoton(T, p['aT'], p['cT'], p['sT']); gM = monoton(M, p['aM'], p['cM'], p['sM'])
        if self.abl == 'abl_min':
            k = p['karisim'][:, None, :]
            sinir = k[..., 0] * gT + k[..., 1] * gM + k[..., 2] * gT * gM
        else:
            sinir = torch.minimum(gT, gM)
        if self.vpd:     # Liebig: min(g_T, g_M, g_D); g_D = 1 − monoton(VPD) → atmosfer kurudukça büyüme azalır
            gD = 1 - monoton(X[..., 3], p['aD'], p['cD'], p['sD'])
            sinir = torch.minimum(sinir, gD)
        C = (p['omega'] * gEk * sinir).sum(1).clamp_min(1e-6)
        BAI = G * C * torch.exp(u_i)
        w_hat = (torch.sqrt((B + BAI) / math.pi) - r) * 10                         # mm
        mu = torch.log(w_hat.clamp_min(1e-4))
        return mu, p['log_sigma'], dict(gT=gT, gM=gM, C=C, G=G, lam=p.get('lam'))

class SerbestAg(nn.Module):
    """Kısıtsız karşılaştırma: 24 aylık iklim -> zamansal evrişim; + log yarıçap + alan gömmesi -> log w"""
    def __init__(self, emb=8, gizli=40, r0_bas=2.0, ortam=False):
        super().__init__()
        self.ortam = OrtamGirdi() if ortam else None
        self.conv = nn.Sequential(nn.Conv1d(5, 32, 3), nn.GELU(), nn.Conv1d(32, 32, 3), nn.GELU())
        self.ge = nn.Embedding(NG + 1, emb); self.se = nn.Embedding(NS + 1, emb)
        self.mlp = nn.Sequential(nn.Linear(32 * 20 + 2 * emb + DESC.shape[1] + 1 + (ORTAM_BOYUT if ortam else 0), gizli), nn.GELU(),
                                 nn.Linear(gizli, gizli), nn.GELU(), nn.Linear(gizli, 2))
        self.gz = Gizil(r0_bas)
    def forward(self, idx, gizle=None, ornek=None, G=None):
        G = girdiler(idx) if G is None else G
        X = (G['X'] - oz_ort) / oz_ss
        h = self.conv(X.transpose(1, 2)).flatten(1)
        g, sp = G['g'], G['sp']
        if gizle is not None:
            g = torch.where(gizle, torch.full_like(g, NG), g); sp = torch.where(gizle, torch.full_like(sp, NS), sp)
        lr = torch.log1p(G['rp'])[:, None]
        d = G['desc'] + (AYAR['desc_gurultu'] * torch.randn_like(G['desc']) if self.training else 0)
        parcalar = [h, self.ge(g), self.se(sp), d, lr] + ([self.ortam(G)] if self.ortam is not None else [])
        o = self.mlp(torch.cat(parcalar, 1))
        _, u_i = self.gz.gizil_deger(G['ser'], ornek)
        if not AYAR['rastgele_etki']: u_i = torch.zeros_like(u_i)
        return o[:, 0] - 0.3 + u_i, o[:, 1] - 0.3, {}

def yap(ad):
    """Model adı: taban ad + isteğe bağlı '_ortam' eki (yetişme ortamı girdileri)"""
    ortam = ad.endswith('_ortam'); ad = ad[:-6] if ortam else ad
    T = lambda **k: TeoriModeli(AYAR['J'], AYAR['emb'], AYAR['gizli'], k.pop('abl', None), AYAR['r0_bas_cm'], ortam=ortam, **k).to(DEV)
    if ad == 'serbest': return SerbestAg(AYAR['emb'], r0_bas=AYAR['r0_bas_cm'], ortam=ortam).to(DEV)
    if ad == 'teori': return T()
    if ad == 'teori_vpd': return T(vpd=True)
    if ad == 'teori_uyum': return T(uyum=True)
    if ad == 'teori_genis': return T(genis=True)
    if ad == 'teori_tur': return T(tur=True)
    if ad == 'teori_saf': return T(vpd=True, saf=True)
    if ad.startswith('teori_mf'):       # profil olabilirlik: teori_mf050 -> m = 0.50 sabit
        return T(m_sabit=int(ad[8:]) / 1000 if len(ad) == 12 else int(ad[8:]) / 100)
    if ad.startswith('teori_m'): return T(m_bas=int(ad[7:]) / 100)
    if ad.startswith('abl_'): return T(abl=ad)
    raise ValueError(ad)

def ag_param_sayisi(m):
    return sum(p.numel() for n, p in m.named_parameters() if not n.startswith('gz.'))

for ad in ['teori_ortam', 'teori_tur']:
    print(f'{ad:16s} eğitilebilir ağ parametresi: {ag_param_sayisi(yap(ad)):,}')

In [ ]:
# --- Eğitim ---
LOG_Y_SIFIR = math.log(AYAR['sifir_mm'])
def hedef(idx):
    w = R_w[idx]
    return torch.where(w > 0, torch.log(w.clamp_min(1e-6)), torch.full_like(w, LOG_Y_SIFIR))

LOG_ESIK = math.log(0.01)            # ölçüm çözünürlüğü: sıfır halka = "0,01 mm'den küçük" (sansürlü gözlem)
def nll(mu, ls, y):
    sifir = y <= LOG_Y_SIFIR + 1e-6
    tam = 0.5 * ((y - mu) / torch.exp(ls)) ** 2 + ls + 0.9189385
    sans = -torch.special.log_ndtr((LOG_ESIK - mu) / torch.exp(ls))
    return torch.where(sifir, sans, tam).mean()

@torch.no_grad()
def tahmin(model, idx, seri_gor, agac_gor, gizle_alan=False, K=8, bs=131072):
    """Görülmüş ağaçlar: koşullu öngörü. Görülmemiş ağaçlar (yeni alan/tür): ρ ve u önselden K kez çekilir;
    öngörü = karışımın ortalaması, varyans = ort(σ²) + var(μ)  (log ölçeğinde)"""
    model.eval(); MU, SG = [], []
    g_gor = torch.unique(site_genus[ser_site[seri_gor.nonzero().squeeze(1)]])
    mu_glob = model.gz.mu_g[g_gor].mean()
    with torch.no_grad():
        for i in range(0, len(idx), bs):
            b = idx[i:i+bs]
            gz = torch.ones(len(b), dtype=torch.bool, device=DEV) if gizle_alan else None
            yeni = ~agac_gor[ser_tree[R_ser[b]]] | ~seri_gor[R_ser[b]]
            if not AYAR['rastgele_etki']: yeni = torch.zeros_like(yeni)
            if not yeni.any():
                mu, ls, _ = model(b, gz); MU.append(mu); SG.append(torch.exp(ls)); continue
            ms, vs = [], []
            for k in range(K):
                mu, ls, _ = model(b, gz, ornek=(seri_gor, agac_gor, gz, mu_glob)); ms.append(mu); vs.append(torch.exp(2 * ls))
            ms, vs = torch.stack(ms), torch.stack(vs)
            MU.append(ms.mean(0)); SG.append(torch.sqrt(vs.mean(0) + ms.var(0, unbiased=False)))
    model.train()
    return torch.cat(MU), torch.cat(SG)

def kat_gor_ayarla(egt):
    """Eğitim halkalarının alanlarında görülen biyom / ekolojik bölge / Köppen sınıfları"""
    sg = torch.unique(ser_site[R_ser[egt]]); KAT_GOR.clear()
    for anahtar, dizi, n in [('b', site_biyom, NB), ('e', site_eko, NE), ('k', site_kg, NK)]:
        m = torch.zeros(n + 1, dtype=torch.bool, device=DEV); m[dizi[sg]] = True; KAT_GOR[anahtar] = m

def egit(bolunme_ad, model_ad, tohum):
    is_ad = f'{bolunme_ad.replace(":", "-")}__{model_ad}__s{tohum}'
    klasor = os.path.join(SONUC, is_ad); os.makedirs(klasor, exist_ok=True)
    if os.path.exists(os.path.join(klasor, 'bitti.json')):
        print('atlandı (bitmiş):', is_ad); return
    torch.manual_seed(tohum); np.random.seed(tohum)
    egt, dog, test, gizle = bolunme(bolunme_ad)
    kat_gor_ayarla(egt)
    if AYAR['hizli_deneme']:
        egt, dog, test = egt[torch.randperm(len(egt), device=DEV)[:200000]], dog[:20000], test[:20000]
    model = yap(model_ad)
    gizil = [p for n, p in model.named_parameters() if n.startswith('gz.')]
    ag = [p for n, p in model.named_parameters() if not n.startswith('gz.')]
    opt = torch.optim.AdamW([{'params': ag, 'lr': AYAR['lr'], 'weight_decay': AYAR['wd']},
                             {'params': gizil, 'lr': AYAR['lr_gizil'] if AYAR['rastgele_etki'] else 0.0, 'weight_decay': 0.0}])
    kayit = os.path.join(klasor, 'son.pt'); bas_ep, en_iyi, sabir, gecmis = 0, float('inf'), 0, []
    if os.path.exists(kayit):
        ck = torch.load(kayit, map_location=DEV)
        model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt'])
        bas_ep, en_iyi, sabir, gecmis = ck['epoch'] + 1, ck['en_iyi'], ck['sabir'], ck['gecmis']
        print(f'{is_ad}: {bas_ep}. epoch\'tan devam')
    n_egt = len(egt)
    agac_gor = torch.zeros(NTREE, dtype=torch.bool, device=DEV); agac_gor[ser_tree[R_ser[egt]]] = True
    seri_gor = torch.zeros(NSER, dtype=torch.bool, device=DEV); seri_gor[R_ser[egt]] = True
    for ep in range(bas_ep, AYAR['max_epoch']):
        t0 = time.time(); kar = egt[torch.randperm(n_egt, device=DEV)]; top = 0.
        for i in range(0, n_egt, AYAR['batch']):
            b = kar[i:i+AYAR['batch']]
            gz = (torch.rand(len(b), device=DEV) < AYAR['takson_dusur'])
            mu, ls, _ = model(b, gz)
            kayip_v = nll(mu, ls, hedef(b))
            kayip = kayip_v + (model.gz.ceza(seri_gor, agac_gor) / n_egt if AYAR['rastgele_etki'] else 0.)
            opt.zero_grad(set_to_none=True); kayip.backward()
            torch.nn.utils.clip_grad_norm_(ag, 5.0); opt.step(); top += kayip_v.item() * len(b)
        model.gz.gorgul_guncelle(seri_gor, agac_gor)
        mu, sg = tahmin(model, dog, seri_gor, agac_gor)
        yd = hedef(dog); v = nll(mu, torch.log(sg), yd).item()
        nz = yd > LOG_Y_SIFIR + 1e-6; v_rmse = torch.sqrt(((mu - yd)[nz] ** 2).mean()).item(); v_sg = torch.sqrt((sg ** 2).mean()).item()
        gecmis.append(dict(epoch=ep, egitim=top / n_egt, dogrulama=v, sn=time.time() - t0))
        ek = (f'  σ_alan {model.gz.sv_gorgul.item():.2f}  σ_ağaç {model.gz.su().item():.2f}' if AYAR['rastgele_etki'] else '')
        print(f'{is_ad} ep {ep:2d}  eğitim {top/n_egt:.4f}  doğrulama {v:.4f} (RMSE {v_rmse:.3f}, öngörü σ {v_sg:.3f}){ek}  ({time.time()-t0:.0f} sn)')
        if v < en_iyi - 1e-4:
            en_iyi, sabir = v, 0; torch.save(model.state_dict(), os.path.join(klasor, 'en_iyi.pt'))
        else:
            sabir += 1
        torch.save(dict(model=model.state_dict(), opt=opt.state_dict(), epoch=ep, en_iyi=en_iyi, sabir=sabir, gecmis=gecmis), kayit)
        if sabir >= AYAR['sabir']: break
    # Test tahminleri (en iyi model)
    model.load_state_dict(torch.load(os.path.join(klasor, 'en_iyi.pt'), map_location=DEV))
    mu, sg = tahmin(model, test, seri_gor, agac_gor, gizle_alan=gizle)
    y = hedef(test)
    np.savez_compressed(os.path.join(klasor, 'test_tahmin.npz'), idx=test.cpu().numpy().astype(np.int32),
                        mu=mu.cpu().numpy().astype(np.float16), sigma=sg.cpu().numpy().astype(np.float16))
    nz = y > LOG_Y_SIFIR + 1e-6         # nokta ölçütleri sıfır olmayan halkalarda; NLL sansürlü olanları da içerir
    ozet = dict(is_ad=is_ad, n_egitim=n_egt, n_test=len(test), rmse_log=float(torch.sqrt(((mu - y)[nz] ** 2).mean())),
                bias_log=float((mu - y)[nz].mean()), nll=float(nll(mu, torch.log(sg), y)),
                kapsama90=float(((y - mu).abs() < 1.645 * sg)[nz].float().mean()), en_iyi_dogrulama=en_iyi,
                ag_param=ag_param_sayisi(model), gecmis=gecmis, ayar=AYAR)
    if not model_ad.startswith('serbest'):           # alan parametreleri (m, eşikler, ω) — şekiller için
        with torch.no_grad():
            p = model.parametreler(torch.arange(S, device=DEV), torch.full((S,), gizle, device=DEV) & t(site_takson, torch.bool))
        np.savez_compressed(os.path.join(klasor, 'alan_param.npz'), **{k: v.cpu().numpy() for k, v in p.items()},
                            r0_cm=F.softplus(model.gz.rho).detach().cpu().numpy(), u=model.gz.u.detach().cpu().numpy())
    json.dump(ozet, open(os.path.join(klasor, 'bitti.json'), 'w'), indent=1)
    os.remove(kayit)
    print(f'BİTTİ {is_ad}: test RMSE(log) {ozet["rmse_log"]:.3f}, kapsama %90 → {ozet["kapsama90"]:.3f}')

In [ ]:
# --- Ortak araçlar: topluluk tahmini, ölçütler, bootstrap ---
import glob, pandas as pd
from scipy.stats import norm
ANALIZ = os.path.join(ANA, 'ANALIZ_SON'); os.makedirs(ANALIZ, exist_ok=True)
Yn = hedef(torch.arange(NR, device=DEV)).cpu().numpy(); NZn = (R_w > 0).cpu().numpy()
SITE_n = ser_site[R_ser].cpu().numpy(); YIL_n = R_yil.cpu().numpy()
def topluluk(b, m):
    MU, V, idx = [], [], None
    for kl in sorted(glob.glob(os.path.join(SONUC, f'{b}__{m}__s*'))):
        if not os.path.exists(os.path.join(kl, 'bitti.json')): continue
        z = np.load(os.path.join(kl, 'test_tahmin.npz'))
        if idx is None: idx = z['idx']
        MU.append(z['mu'].astype(np.float32)); V.append(z['sigma'].astype(np.float32) ** 2)
    if not MU: return None
    MU, V = np.stack(MU), np.stack(V)
    return idx, MU.mean(0), np.sqrt(V.mean(0) + MU.var(0)), len(MU)
def olcutler(y, mu, sg, nz):
    r = (y - mu)[nz]; z = (y - mu) / sg
    nllv = np.where(nz, 0.5 * z**2 + np.log(sg) + 0.9189385, -norm.logcdf((math.log(0.01) - mu) / sg))
    crps = sg * (z * (2 * norm.cdf(z) - 1) + 2 * norm.pdf(z) - 1 / math.sqrt(math.pi))
    return dict(RMSE=float(np.sqrt((r**2).mean())), bias=float(r.mean()), NLL=float(nllv.mean()),
                CRPS=float(crps[nz].mean()), kapsama90=float((np.abs(z) < 1.645)[nz].mean()))
rng = np.random.default_rng(2026)
def boot_fark(y, ma, mb, nz, grup, B=1000):
    """RMSE(a) − RMSE(b) ve grup (alan ya da yıl) bootstrap aralıkları"""
    g = grup[nz]; ea, eb = (y - ma)[nz] ** 2, (y - mb)[nz] ** 2
    u, inv = np.unique(g, return_inverse=True)
    Sa, Sb, n = np.bincount(inv, ea, len(u)), np.bincount(inv, eb, len(u)), np.bincount(inv, None, len(u))
    W = rng.multinomial(len(u), np.full(len(u), 1 / len(u)), size=B).astype(float)
    d = np.sqrt(W @ Sa / (W @ n)) - np.sqrt(W @ Sb / (W @ n))
    return float(np.sqrt(Sa.sum() / n.sum()) - np.sqrt(Sb.sum() / n.sum())), np.quantile(d, [0.025, 0.975]), np.quantile(d, [0.05, 0.95])
def alan_agirlikli_rmse(y, mu, nz, s):
    u, inv = np.unique(s[nz], return_inverse=True)
    return float(np.sqrt(np.bincount(inv, (y - mu)[nz] ** 2) / np.bincount(inv)).mean())

In [ ]:
# --- Senaryo altyapısı (Defter 5–6 ile aynı) ---
CL = d['clim']; LAT = d['site_lat']
def gun_uzunlugu(lat):
    gun = np.array([15, 46, 74, 105, 135, 166, 196, 227, 258, 288, 319, 349])
    dek = 23.45 * np.sin(np.deg2rad(360 * (284 + gun) / 365))
    x = -np.tan(np.deg2rad(lat))[:, None] * np.tan(np.deg2rad(dek))[None, :]
    return 24 / np.pi * np.arccos(np.clip(x, -1, 1))
def isi_indeksi(T):
    S_ = T.shape[0]; Tclim = np.nanmean(T.reshape(S_, -1, 12), axis=1)
    return np.sum(np.where(Tclim > 0, (np.maximum(Tclim, 0) / 5) ** 1.514, 0), axis=1)
def kova(T, P, lat, I, Mmax=0.76, Mmin=0.01, alph=0.093, m_th=4.886, mu_th=5.80, rootd=1000., M0=0.2):
    """Defter 1 ile aynı VS-Lite kova modeli; ısı indeksi I dışarıdan (taban iklim) verilir"""
    S_, N = T.shape; gu = gun_uzunlugu(lat)
    ngun = np.array([31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]); L = (ngun / 30)[None, :] * (gu / 12)
    a = 6.75e-7 * I**3 - 7.71e-5 * I**2 + 1.792e-2 * I + 0.49
    M = np.empty((S_, N), np.float32); m = np.full(S_, M0)
    for t in range(N):
        Tt, Pt, Lt = T[:, t], P[:, t], L[:, t % 12]
        Ep = np.where(Tt <= 0, 0., np.where(Tt < 26.5, 16 * Lt * (10 * np.maximum(Tt, 0) / np.maximum(I, 1e-6)) ** a, -415.85 + 32.25 * Tt - 0.43 * Tt**2))
        Ep = np.maximum(Ep, 0); E = Ep * (m / Mmax)
        G_ = mu_th * alph / (1 + mu_th) * m * rootd; R_ = Pt * (m / Mmax) ** m_th + (alph / (1 + mu_th)) * m * rootd
        m = np.clip(m + (Pt - E - R_ - G_) / rootd, Mmin, Mmax); M[:, t] = m
    return M
es = lambda T: 0.6108 * np.exp(17.27 * T / (T + 237.3))
@torch.no_grad()
def tahmin_G(model, G, bs=65536):
    n = len(G['rp']); out = []
    for i in range(0, n, bs): out.append(model(None, None, G={k: v[i:i+bs] for k, v in G.items()})[0])
    return torch.cat(out)
def yukle(bolunme_ad, model_ad, tohum=0):
    kl = os.path.join(SONUC, f'{bolunme_ad.replace(":", "-")}__{model_ad}__s{tohum}')
    if not os.path.exists(os.path.join(kl, 'en_iyi.pt')): return None, None
    egt, _, test, _ = bolunme(bolunme_ad); kat_gor_ayarla(egt)
    m = yap(model_ad); m.load_state_dict(torch.load(os.path.join(kl, 'en_iyi.pt'), map_location=DEV)); m.eval()
    return m, test
gen = torch.Generator(device='cpu').manual_seed(0)
_, _, test_z, _ = bolunme('zaman')
sec = test_z[torch.randperm(len(test_z), generator=gen)[:50000].to(DEV)]
siteler = torch.unique(ser_site[R_ser[sec]]).cpu().numpy()
T0 = CL[siteler, :, :, 0].reshape(len(siteler), -1).astype(np.float64); P0 = CL[siteler, :, :, 1].reshape(len(siteler), -1).astype(np.float64)
V0 = CL[siteler, :, :, 3].reshape(len(siteler), -1).astype(np.float64); I0 = isi_indeksi(T0)
M_taban = kova(T0, P0, LAT[siteler], I0)
M_yagis = kova(T0, P0 * 1.2, LAT[siteler], I0)
M_sicak = kova(T0 + 1, P0, LAT[siteler], I0)
V_sicak = np.maximum(es(T0 + 1) - (es(T0) - V0), 0)
OZ_yedek = OZ[torch.as_tensor(siteler, device=DEV)].clone()
si = torch.as_tensor(siteler, device=DEV)
def ayarla(T=None, P=None, M=None, V=None):
    OZ[si] = OZ_yedek
    if T is not None: OZ[si, :, 0] = t(T, torch.float32)
    if P is not None: OZ[si, :, 1] = t(np.log1p(np.maximum(P, 0)), torch.float32)
    if M is not None: OZ[si, :, 2] = t(M, torch.float32)
    if V is not None: OZ[si, :, 3] = t(V, torch.float32)
s_sec = ser_site[R_ser[sec]].cpu().numpy()
kur = -desc[s_sec, 5]; sog = -desc[s_sec, 2]
kurak_q = kur >= np.quantile(kur, 0.75); soguk_q = sog >= np.quantile(sog, 0.75)
def oranlar(dP, dT):
    return dict(yagis_artan=float((dP > 1e-3).mean()), yagis_tepkisiz=float((np.abs(dP) <= 1e-3).mean()), yagis_azalan_IHLAL=float((dP < -1e-3).mean()),
                sicak_artan=float((dT > 1e-3).mean()), sicak_tepkisiz=float((np.abs(dT) <= 1e-3).mean()), sicak_azalan=float((dT < -1e-3).mean()),
                sicak_azalan_en_kurak_ceyrek=float((dT[kurak_q] < -1e-3).mean()), sicak_azalan_en_soguk_ceyrek=float((dT[soguk_q] < -1e-3).mean()),
                sicak_artan_en_soguk_ceyrek=float((dT[soguk_q] > 1e-3).mean()))

In [ ]:
# --- Özellik fonksiyonları ---
ANA_BOL = ['zaman', 'takson', 'bolge:North America', 'bolge:Europe', 'bolge:Asia', 'bolge:Southern Hemisphere', 'iklim:sicak', 'iklim:kurak']
from scipy.stats import spearmanr
import xgboost as xgb
print('xgboost', xgb.__version__)
NG_SEC = 2_000_000   # XGBoost eğitim alt örneklemi (halka); doğrulama 300 bin
def ozellik_ag(b, gizle):
    """[B, 126]: 24 ay × (T, logP, M, VPD) + log yarıçap + g_E ağırlıklı mevsim ortalamaları (önceki/bu yıl) + alan + cins, tür"""
    X, s = pencere(b); w = X[..., 4]
    mev = [((X[:, a:a+12, c] * w[:, a:a+12]).sum(1) / w[:, a:a+12].sum(1).clamp_min(1e-6))[:, None] for a in (0, 12) for c in range(4)]
    g = site_genus[s].float(); sp = site_species[s].float()
    if gizle: g = torch.full_like(g, float('nan')); sp = torch.full_like(sp, float('nan'))
    return torch.cat([X[..., :4].flatten(1), torch.log(R_rp[b] + 0.1)[:, None]] + mev + [DESC[s], ENV[s], g[:, None], sp[:, None]], 1)
def ozellik_np(idx, gizle, bs=500_000):
    return np.concatenate([ozellik_ag(idx[i:i+bs], gizle).cpu().numpy() for i in range(0, len(idx), bs)]).astype(np.float32)
FT = ['q'] * (96 + 1 + 8 + DESC.shape[1] + ENV.shape[1]) + ['c', 'c']
XDEV = 'cuda' if DEV == 'cuda' else 'cpu'
def kaydet(b, ad, idx, mu, sg, ek):
    kl = os.path.join(SONUC, f'{b.replace(":", "-")}__{ad}__s0'); os.makedirs(kl, exist_ok=True)
    np.savez_compressed(os.path.join(kl, 'test_tahmin.npz'), idx=idx.astype(np.int32), mu=mu.astype(np.float16), sigma=sg.astype(np.float16))
    json.dump(ek, open(os.path.join(kl, 'bitti.json'), 'w'))
def bitti(b, ad): return os.path.exists(os.path.join(SONUC, f'{b.replace(":", "-")}__{ad}__s0', 'bitti.json'))

def dkm_ozellik(b, gizle, g_gor):
    """Doğrusal karma model tasarımı: sabit, log r, (log r)², 8 mevsim iklimi, DESC, ENV, cins (gizlide görülen cinslerin ortalaması)"""
    X, s = pencere(b); w = X[..., 4]
    mev = [((X[:, a:a+12, c] * w[:, a:a+12]).sum(1) / w[:, a:a+12].sum(1).clamp_min(1e-6))[:, None] for a in (0, 12) for c in range(4)]
    lr = torch.log(R_rp[b] + 0.1)[:, None]
    oh = F.one_hot(site_genus[s], NG + 1).float()
    if gizle: oh = g_gor.float()[None, :].expand(len(b), -1) / g_gor.sum()
    return torch.cat([torch.ones_like(lr), lr, lr ** 2] + mev + [DESC[s], ENV[s], oh], 1), s

In [ ]:
# --- Klasik modeller: yeniden kullanılabilir kurulum ve tahmin fonksiyonları ---
XGB_VARSAYILAN = dict(max_depth=10, eta=0.08, subsample=0.8, colsample_bytree=0.6, max_bin=128, min_child_weight=50)
def xgb_kur(b, prm=XGB_VARSAYILAN, n_egt=NG_SEC, n_tur=2000, nicel=True, tohum=0):
    egt, dog, test, gizle = bolunme(b)
    egt = egt[R_w[egt] > 0]; dog = dog[R_w[dog] > 0]
    g = torch.Generator(device=DEV).manual_seed(tohum)
    egt = egt[torch.randperm(len(egt), device=DEV, generator=g)[:n_egt]]; dog = dog[torch.randperm(len(dog), device=DEV, generator=g)[:300_000]]
    if AYAR['hizli_deneme']: egt, dog, test, n_tur = egt[:100_000], dog[:20_000], test[:20_000], 40
    dtr = xgb.DMatrix(ozellik_np(egt, False), hedef(egt).cpu().numpy(), feature_types=FT, enable_categorical=True)
    dva = xgb.DMatrix(ozellik_np(dog, False), hedef(dog).cpu().numpy(), feature_types=FT, enable_categorical=True)
    ort = dict(device=XDEV, tree_method='hist', max_cat_to_onehot=1, seed=tohum, **prm)
    m1 = xgb.train(dict(ort, objective='reg:squarederror'), dtr, n_tur, evals=[(dva, 'dog')], early_stopping_rounds=50, verbose_eval=False)
    m2 = None
    if nicel:
        m2 = xgb.train(dict(ort, objective='reg:quantileerror', quantile_alpha=np.array([0.05, 0.95])), dtr, min(n_tur, 600),
                       evals=[(dva, 'dog')], early_stopping_rounds=50, verbose_eval=False)
    v = float(np.sqrt(np.mean((m1.predict(dva, iteration_range=(0, m1.best_iteration + 1)) - dva.get_label()) ** 2)))
    del dtr, dva
    return dict(m1=m1, m2=m2, test=test, gizle=gizle, dog_rmse=v)
import gc
def xgb_tahmin(K, idx, gizle=False):
    MU, SG = [], []
    for i in range(0, len(idx), 1_000_000):
        dte = xgb.DMatrix(ozellik_np(idx[i:i+1_000_000], gizle), feature_types=FT, enable_categorical=True)
        MU.append(K['m1'].predict(dte, iteration_range=(0, K['m1'].best_iteration + 1)))
        if K['m2'] is not None:
            q = K['m2'].predict(dte, iteration_range=(0, K['m2'].best_iteration + 1)); SG.append(np.maximum((q[:, 1] - q[:, 0]) / 3.29, 0.05))
    return np.concatenate(MU), (np.concatenate(SG) if SG else None)
def dkm_kur(b, bs=400_000):
    egt, dog, test, gizle = bolunme(b); egt = egt[R_w[egt] > 0]
    if AYAR['hizli_deneme']: egt, test = egt[:200_000], test[:20_000]
    g_gor = torch.zeros(NG + 1, dtype=torch.bool, device=DEV); g_gor[site_genus[ser_site[R_ser[egt]]]] = True
    y_all = hedef(egt); site_e = ser_site[R_ser[egt]]; u = torch.zeros(S, device=DEV)
    for it in range(3):
        XtX = 0; Xty = 0
        for i in range(0, len(egt), bs):
            Z, s = dkm_ozellik(egt[i:i+bs], False, g_gor); rr = y_all[i:i+bs] - u[s]
            XtX = XtX + Z.T.double() @ Z.double(); Xty = Xty + Z.T.double() @ rr.double()
        p_ = XtX.shape[0]; beta = torch.linalg.solve(XtX + 1e-3 * torch.eye(p_, device=DEV, dtype=torch.float64) * XtX.diagonal().mean(), Xty).float()
        res = torch.cat([y_all[i:i+bs] - dkm_ozellik(egt[i:i+bs], False, g_gor)[0] @ beta for i in range(0, len(egt), bs)])
        n_s = torch.bincount(site_e, minlength=S).float(); top = torch.bincount(site_e, res, minlength=S)
        gor = n_s > 0; rbar = top[gor] / n_s[gor]
        s2e = float(((res - (top / n_s.clamp_min(1))[site_e]) ** 2).sum() / (len(res) - gor.sum()))
        s2u = max(float(rbar.var() - (s2e / n_s[gor]).mean()), 1e-4)
        u = torch.zeros(S, device=DEV); u[gor] = n_s[gor] * s2u / (n_s[gor] * s2u + s2e) * rbar
    return dict(beta=beta, u=u, n_s=n_s, g_gor=g_gor, s2e=s2e, s2u=s2u, test=test, gizle=gizle)
@torch.no_grad()
def dkm_tahmin(K, idx, gizle=False, bs=400_000):
    MU = []
    for i in range(0, len(idx), bs):
        Z, s = dkm_ozellik(idx[i:i+bs], gizle, K['g_gor']); MU.append((Z @ K['beta'] + K['u'][s]).cpu().numpy())
    return np.concatenate(MU)

In [ ]:
# --- G1: tüm modellerin senaryo tepkileri (XGBoost ve karma model dahil) ve G2 için halka düzeyi değişimler ---
G12_BITTI = os.path.exists(os.path.join(ANALIZ, 'G1_senaryo_tum_modeller.csv')) and os.path.exists(os.path.join(ANALIZ, 'G2_ampirik_yon_uyumu.csv'))
if G12_BITTI: print('G1 ve G2 daha önce tamamlandı — atlanıyor')
DEG = {}   # model -> (dP, dT) halka başına log büyüme değişimi
for m_ad in ([] if G12_BITTI else ['serbest', 'teori', 'serbest_ortam', 'teori_ortam', 'teori_vpd_ortam', 'teori_uyum_ortam', 'teori_tur', 'abl_mono_ortam']):
    Y0, Y1, Y2 = [], [], []
    for tohum in range(5):
        model, _ = yukle('zaman', m_ad, tohum)
        if model is None: continue
        ayarla(M=M_taban); Y0.append(tahmin_G(model, girdiler(sec)).cpu().numpy())
        ayarla(P=P0 * 1.2, M=M_yagis); Y1.append(tahmin_G(model, girdiler(sec)).cpu().numpy())
        ayarla(T=T0 + 1, M=M_sicak, V=V_sicak); Y2.append(tahmin_G(model, girdiler(sec)).cpu().numpy())
        del model; torch.cuda.empty_cache()
    if Y0:
        y0, y1, y2 = np.mean(Y0, 0), np.mean(Y1, 0), np.mean(Y2, 0); DEG[m_ad] = (y1 - y0, y2 - y0)
ayarla()
_, _, test_z, _ = bolunme('zaman'); kat_gor_ayarla(bolunme('zaman')[0])
KX = KD = None
if not G12_BITTI: KX = xgb_kur('zaman'); KD = dkm_kur('zaman')
for ad, f in ([] if G12_BITTI else  [('xgb', lambda: xgb_tahmin(KX, sec)[0]), ('dkm', lambda: dkm_tahmin(KD, sec))]):
    ayarla(M=M_taban); y0 = f(); ayarla(P=P0 * 1.2, M=M_yagis); y1 = f(); ayarla(T=T0 + 1, M=M_sicak, V=V_sicak); y2 = f()
    DEG[ad] = (y1 - y0, y2 - y0)
ayarla()
if not G12_BITTI:
    g1 = pd.DataFrame([dict(model=m, **oranlar(*DEG[m])) for m in DEG])
    g1.to_csv(os.path.join(ANALIZ, 'G1_senaryo_tum_modeller.csv'), index=False)
else:
    g1 = pd.read_csv(os.path.join(ANALIZ, 'G1_senaryo_tum_modeller.csv'))
print(g1.round(3).to_string(index=False)); del KX, KD; import gc; gc.collect()

In [ ]:
# --- G2: ampirik karşılaştırma — gözlenen iklim–büyüme duyarlılığı ile model tepkisinin yönü ---
# Gözlenen: her serinin log halka genişliğinden 31 yıllık hareketli ortalama çıkarılır (boyut/yaş eğilimi), alan-yıl ortalaması = kronoloji.
# İklim: g_E ağırlıklı yıllık (bu yıl) sıcaklık ve log yağış; 31 yıllık hareketli ortalamadan sapma. Alan başına çoklu regresyon (≥30 yıl).
if not G12_BITTI:
    alanlar_g2 = np.unique(s_sec)
    mask = np.isin(SITE_n, alanlar_g2) & (R_w.cpu().numpy() > 0)
    df = pd.DataFrame(dict(ser=R_ser.cpu().numpy()[mask], s=SITE_n[mask], yil=YIL_n[mask], y=np.log(R_w.cpu().numpy()[mask])))
    df = df.sort_values(['ser', 'yil']).reset_index(drop=True)
    # seri içinde ortalanmış 31 yıllık hareketli ortalama (vektörel; kümülatif toplamla)
    serv = df.ser.values; yv = df.y.values; n_ = len(df); pos = np.arange(n_)
    bas_ = np.r_[0, np.flatnonzero(np.diff(serv)) + 1]; uzun = np.diff(np.r_[bas_, n_])
    sb = np.repeat(bas_, uzun); se_ = sb + np.repeat(uzun, uzun)
    lo = np.maximum(sb, pos - 15); hi = np.minimum(se_, pos + 16)
    cs = np.r_[0, np.cumsum(yv)]; say = hi - lo
    df['iz'] = np.where(say >= 15, yv - (cs[hi] - cs[lo]) / say, np.nan)
    kron = df.dropna().groupby(['s', 'yil']).iz.mean().reset_index()
    w = OZ[:, :, 4].reshape(S, NY, 12); Tm = OZ[:, :, 0].reshape(S, NY, 12); Pm = OZ[:, :, 1].reshape(S, NY, 12)
    Tg = ((Tm * w).sum(2) / w.sum(2).clamp_min(1e-6)).cpu().numpy(); Pg = np.log(np.expm1(Pm.cpu().numpy()).sum(2) + 1)
    def sapma(A):
        k_ = np.ones(31) / 31; out = np.full_like(A, np.nan)
        for i in range(A.shape[0]):
            c = np.convolve(A[i], k_, mode='same'); n = np.convolve(np.ones(A.shape[1]), k_, mode='same'); out[i] = A[i] - c / n
        return out
    Ta, Pa = sapma(Tg), sapma(Pg)
    kron['Ta'] = Ta[kron.s, kron.yil - YIL0]
    kron['Pa'] = Pa[kron.s, kron.yil - YIL0]
    def alan_reg(g):
        g = g.dropna()
        bos = pd.Series(dict(bT=np.nan, bP=np.nan, tT=np.nan, tP=np.nan, n=len(g)))
        if len(g) < 30 or g.Ta.std() < 1e-6 or g.Pa.std() < 1e-6: return bos
        X = np.c_[np.ones(len(g)), g.Ta, g.Pa]; y = g.iz.values
        if np.linalg.cond(X.T @ X) > 1e10: return bos
        XtX = np.linalg.inv(X.T @ X); b = XtX @ X.T @ y; r = y - X @ b; s2 = r @ r / (len(g) - 3); se = np.sqrt(np.diag(XtX) * s2)
        return pd.Series(dict(bT=b[1], bP=b[2], tT=b[1] / se[1], tP=b[2] / se[2], n=len(g)))
    gozlem = kron.groupby('s').apply(alan_reg).dropna()
    print('gözlenen duyarlılık hesaplanan alan:', len(gozlem), '| anlamlı (|t|>2) sıcaklık:', int((gozlem.tT.abs() > 2).sum()), 'yağış:', int((gozlem.tP.abs() > 2).sum()))
    kurak_alan = -desc[:, 5]; soguk_alan = -desc[:, 2]
    qk = np.quantile(kurak_alan[alanlar_g2], 0.75); qs = np.quantile(soguk_alan[alanlar_g2], 0.75)
    g2 = []
    for m_ad, (dP, dT) in DEG.items():
        md = pd.DataFrame(dict(s=s_sec, dP=dP, dT=dT)).groupby('s').mean()
        j = gozlem.join(md, how='inner')
        for kume, sec_ in [('tüm alanlar', np.ones(len(j), bool)), ('en soğuk çeyrek', soguk_alan[j.index] >= qs), ('en kurak çeyrek', kurak_alan[j.index] >= qk)]:
            jj = j[sec_]
            for deg, b_, t_, dm in [('yağış', 'bP', 'tP', 'dP'), ('sıcaklık', 'bT', 'tT', 'dT')]:
                an = jj[jj[t_].abs() > 2]
                uy = float((np.sign(an[dm]) == np.sign(an[b_])).mean()) if len(an) else np.nan
                from scipy.stats import spearmanr as _sp
                rho = float(_sp(jj[b_], jj[dm])[0]) if len(jj) > 10 else np.nan
                g2.append(dict(model=m_ad, kume=kume, degisken=deg, n_alan=len(jj), n_anlamli=len(an), yon_uyumu_anlamli=uy, spearman_tum=rho,
                               gozlenen_pozitif_orani=float((an[b_] > 0).mean()) if len(an) else np.nan))
    g2 = pd.DataFrame(g2); g2.to_csv(os.path.join(ANALIZ, 'G2_ampirik_yon_uyumu.csv'), index=False)
    gozlem.to_csv(os.path.join(ANALIZ, 'G2b_gozlenen_alan_duyarliligi.csv'))
    print(g2.pivot_table(index='model', columns=['kume', 'degisken'], values='yon_uyumu_anlamli').round(2).to_string())
    print(g2.pivot_table(index='model', columns=['kume', 'degisken'], values='spearman_tum').round(2).to_string())
else:
    print(pd.read_csv(os.path.join(ANALIZ, 'G2_ampirik_yon_uyumu.csv')).round(2).to_string(index=False))

In [ ]:
# --- G3: tohum değişkenliğini de içeren güven aralıkları ve eşdeğerlik sınırı duyarlılığı (±0,01 / 0,02 / 0,03) ---
def uyeler(b, m):
    MU, idx = [], None
    for kl in sorted(glob.glob(os.path.join(SONUC, f'{b}__{m}__s*'))):
        if not os.path.exists(os.path.join(kl, 'bitti.json')): continue
        z = np.load(os.path.join(kl, 'test_tahmin.npz'))
        if idx is None: idx = z['idx']
        MU.append(z['mu'].astype(np.float32))
    return idx, (np.stack(MU) if MU else None)
g3 = []; B3 = 50 if AYAR['hizli_deneme'] else 200
G3_YOL = os.path.join(ANALIZ, 'G3_tohumlu_esdegerlik.csv')
for b in ([] if os.path.exists(G3_YOL) else  ['zaman', 'takson', 'bolge-North America', 'bolge-Europe', 'bolge-Asia', 'bolge-Southern Hemisphere', 'iklim-sicak', 'iklim-kurak']):
    for ma, mb in [('teori', 'serbest'), ('teori_ortam', 'serbest_ortam')]:
        ia, A_ = uyeler(b, ma); ib, B_ = uyeler(b, mb)
        if A_ is None or B_ is None: continue
        y = Yn[ia]; nz = NZn[ia]; st = SITE_n[ia][nz]; u_, inv = np.unique(st, return_inverse=True); ns = np.bincount(inv, minlength=len(u_))
        fark = []
        for k_ in range(B3):
            sa = rng.integers(0, len(A_), len(A_)); sb = rng.integers(0, len(B_), len(B_))
            ea = np.bincount(inv, ((y - A_[sa].mean(0))[nz]) ** 2, len(u_)); eb = np.bincount(inv, ((y - B_[sb].mean(0))[nz]) ** 2, len(u_))
            W = rng.multinomial(len(u_), np.full(len(u_), 1 / len(u_))).astype(float)
            fark.append(np.sqrt(W @ ea / (W @ ns)) - np.sqrt(W @ eb / (W @ ns)))
        lo, hi = np.quantile(fark, [0.05, 0.95])
        sinif = lambda mrg: 'eşdeğer' if (lo > -mrg and hi < mrg) else ('teori daha iyi' if hi < 0 else ('teori daha kötü' if lo > 0 else 'belirsiz'))
        g3.append(dict(bolunme=b, teori=ma, serbest=mb, GA90_alt=lo, GA90_ust=hi, sinir_001=sinif(0.01), sinir_002=sinif(0.02), sinir_003=sinif(0.03)))
    print('tamam:', b)
if g3: pd.DataFrame(g3).to_csv(G3_YOL, index=False)
g3 = pd.read_csv(G3_YOL); print(g3.round(4).to_string(index=False))

In [ ]:
# --- G4: ayarlanmış XGBoost — zamansal testte rastgele arama, en iyi ayarla tüm testlerde daha büyük eğitim kümesi ---
yol_ayar = os.path.join(ANALIZ, 'G4_xgb_ayar_arama.csv')
if os.path.exists(yol_ayar):
    arama = pd.read_csv(yol_ayar)
else:
    r4 = np.random.default_rng(7); arama = []
    for k_ in range(4 if AYAR['hizli_deneme'] else 16):
        prm = dict(max_depth=int(r4.choice([6, 8, 10, 12, 14])), eta=float(r4.choice([0.03, 0.05, 0.08, 0.12])), subsample=float(r4.choice([0.6, 0.8, 1.0])),
                   colsample_bytree=float(r4.choice([0.4, 0.6, 0.8])), max_bin=int(r4.choice([128, 256])), min_child_weight=float(r4.choice([10, 50, 200])),
                   reg_lambda=float(r4.choice([1.0, 5.0, 20.0])))
        K_ = xgb_kur('zaman', prm, n_egt=1_000_000, n_tur=3000, nicel=False)
        arama.append(dict(prm, dog_rmse=K_['dog_rmse'], tur=int(K_['m1'].best_iteration))); print(arama[-1]); del K_
    arama = pd.DataFrame(arama); arama.to_csv(yol_ayar, index=False)
en_iyi = arama.sort_values('dog_rmse').iloc[0]
PRM = {k: (int(en_iyi[k]) if k in ('max_depth', 'max_bin') else float(en_iyi[k])) for k in ['max_depth', 'eta', 'subsample', 'colsample_bytree', 'max_bin', 'min_child_weight', 'reg_lambda']}
print('en iyi ayar:', PRM)
for b in ANA_BOL + ['iklim:kurak2', 'iklim:nemli']:
    if bitti(b, 'xgb_ayar'): print('atlandı:', b); continue
    t0 = time.time(); kat_gor_ayarla(bolunme(b)[0])
    K_ = xgb_kur(b, PRM, n_egt=6_000_000, n_tur=4000)
    mu, sg = xgb_tahmin(K_, K_['test'], K_['gizle'])
    kaydet(b, 'xgb_ayar', K_['test'].cpu().numpy(), mu, sg, dict(prm=PRM, tur=int(K_['m1'].best_iteration), sn=time.time() - t0))
    print(f'xgb_ayar {b}: {time.time()-t0:.0f} sn'); del K_, mu, sg; gc.collect()
g4 = []
for b in ANA_BOL + ['iklim:kurak2', 'iklim:nemli']:
    bb = b.replace(':', '-')
    for m in ['xgb', 'xgb_ayar', 'serbest_ortam', 'teori_ortam']:
        T_ = topluluk(bb, m)
        if T_ is None: continue
        idx, mu, sg, n = T_; g4.append(dict(bolunme=bb, model=m, **olcutler(Yn[idx], mu, sg, NZn[idx])))
g4 = pd.DataFrame(g4); g4.to_csv(os.path.join(ANALIZ, 'G4_xgb_ayarli.csv'), index=False)
print(g4.pivot(index='model', columns='bolunme', values='RMSE').round(3).to_string())

In [ ]:
# --- G5: başlangıç değeri ve yanlış belirtilmiş üretici — m ne kadar sağlam? ---
# (a) Gerçek veri: m başlangıcı 0,40 ve 0,80 (varsayılan 2/3) ile T+S, zamansal test, tohum 0.
for mb in ['teori_m40_ortam', 'teori_m80_ortam']:
    egit('zaman', mb, 0)
# (b) Sentetik veri serbest ağdan (U+S) üretilir: teori modeli "doğru" yapıyı içermeyen veriden hangi m'yi çıkarır?
SEN2 = os.path.join(ANA, 'SONUC_SENTETIK2'); os.makedirs(SEN2, exist_ok=True); yol2 = os.path.join(SEN2, 'Y_sentetik_U.npy')
if not os.path.exists(yol2):
    ure, _ = yukle('zaman', 'serbest_ortam', 0); gen2 = torch.Generator(device=DEV).manual_seed(321); Ys = []
    with torch.no_grad():
        for i in range(0, NR, 262144):
            b_ = torch.arange(i, min(i + 262144, NR), device=DEV); mu, ls, _ = ure(b_)
            yy = mu + torch.exp(ls) * torch.randn(len(b_), device=DEV, generator=gen2)
            Ys.append(torch.where(yy < LOG_ESIK, torch.full_like(yy, LOG_Y_SIFIR), yy).cpu())
    np.save(yol2, torch.cat(Ys).numpy()); del ure
Y_SEN2 = t(np.load(yol2), torch.float32)
hedef_orj, SONUC_orj = hedef, SONUC
hedef = lambda idx: Y_SEN2[idx]; SONUC = SEN2
try:
    egit('zaman', 'teori_ortam', 0)
finally:
    hedef, SONUC = hedef_orj, SONUC_orj
egt_z = bolunme('zaman')[0]; ag_ = np.zeros(S, bool); ag_[np.unique(ser_site[R_ser[egt_z]].cpu().numpy())] = True
def m_oku(kl):
    f = os.path.join(kl, 'alan_param.npz'); return np.load(f)['m'] if os.path.exists(f) else None
M = {'gerçek, başlangıç 2/3 (tohum 0)': m_oku(os.path.join(SONUC, 'zaman__teori_ortam__s0')),
     'gerçek, başlangıç 0,40': m_oku(os.path.join(SONUC, 'zaman__teori_m40_ortam__s0')),
     'gerçek, başlangıç 0,80': m_oku(os.path.join(SONUC, 'zaman__teori_m80_ortam__s0')),
     'sentetik (U+S üretici)': m_oku(os.path.join(SEN2, 'zaman__teori_ortam__s0'))}
gc = site_genus.cpu().numpy(); g5 = []
ref = M['gerçek, başlangıç 2/3 (tohum 0)']
for k_, v in M.items():
    if v is None: continue
    cins = pd.Series(v[ag_]).groupby(gc[ag_]).median()
    g5.append(dict(durum=k_, medyan=float(np.median(v[ag_])), q25=float(np.quantile(v[ag_], 0.25)), q75=float(np.quantile(v[ag_], 0.75)),
                   cins_medyan_araligi=f'{cins.min():.2f}–{cins.max():.2f}',
                   referansla_spearman=(float(spearmanr(v[ag_], ref[ag_])[0]) if ref is not None else np.nan),
                   referansla_ort_mutlak_fark=(float(np.abs(v[ag_] - ref[ag_]).mean()) if ref is not None else np.nan)))
g5 = pd.DataFrame(g5); g5.to_csv(os.path.join(ANALIZ, 'G5_m_saglamlik.csv'), index=False); print(g5.round(3).to_string(index=False))

In [ ]:
# --- G6: en kurak %10 — Kuzey Amerika içi ve dışı ayrı ---
_, _, test_k, _ = bolunme('iklim:kurak')
LON = d['site_lon']; bolge_ad = np.array(soz['site_bolge'])
g6 = []
for m in ['serbest', 'serbest_ortam', 'teori_ortam', 'teori_uyum_ortam', 'teori_genis_ortam', 'abl_mono_ortam', 'dkm', 'xgb', 'xgb_ayar']:
    T_ = topluluk('iklim-kurak', m)
    if T_ is None: continue
    idx, mu, sg, n = T_
    for ad, w_ in [('Kuzey Amerika', bolge_ad[SITE_n[idx]] == 'North America'), ('Kuzey Amerika dışı', bolge_ad[SITE_n[idx]] != 'North America')]:
        if w_.sum() == 0: continue
        g6.append(dict(model=m, alt_kume=ad, n_alan=int(len(np.unique(SITE_n[idx][w_]))), n_halka=int(w_.sum()), **olcutler(Yn[idx][w_], mu[w_], sg[w_], NZn[idx][w_])))
g6 = pd.DataFrame(g6)
T_a, T_b = topluluk('iklim-kurak', 'teori_uyum_ortam'), topluluk('iklim-kurak', 'teori_ortam')
if T_a is not None and T_b is not None:
    idx = T_a[0]; w_ = bolge_ad[SITE_n[idx]] != 'North America'
    if w_.sum():
        f, ga, _ = boot_fark(Yn[idx][w_], T_a[1][w_], T_b[1][w_], NZn[idx][w_], SITE_n[idx][w_])
        print(f'Kuzey Amerika dışı kurak alanlar: uyum − T+S = {f:+.3f} (GA95 {ga[0]:+.3f}, {ga[1]:+.3f})')
g6.to_csv(os.path.join(ANALIZ, 'G6_kurak_bolge.csv'), index=False)
print(g6[['model', 'alt_kume', 'n_alan', 'n_halka', 'RMSE', 'bias', 'kapsama90']].round(3).to_string(index=False))
print('==== DEFTER 7 TAMAM ====  Dosyalar:', ANALIZ)